# Numerical Solution of a Delay Differential Equation (DDE)

## Problem Statement

We study the nonlinear **delay differential equation (DDE)**:

$$h'(u) = -\ln(2)\,(h(u) + 1)\,h(u - 1), \qquad u \in [0,\, U]$$

with **history condition** (the DDE analogue of an initial condition):

$$h(u) = \phi(u), \qquad u \in [-1,\, 0]$$

### What makes this a DDE?

Unlike an ODE, the rate of change `h'(u)` at time `u` depends not only on the *current* value `h(u)` but also on the value **one unit of time in the past**, `h(u − 1)`. This "memory" effect is the defining feature of a delay differential equation and means the entire history on `[−1, 0]` must be prescribed before integration can begin.

### Equilibria

Setting `h'(u) = 0` gives two constant equilibrium solutions:

- **h\*(u) = 0** — the trivial equilibrium
- **h\*(u) = −1** — a non-trivial equilibrium

The experiments in this notebook explore how trajectories starting from different initial histories (`phi = 1` and `phi = −0.5`) converge toward these equilibria over `u ∈ [0, 20]`.

## Numerical Method

### Why DDEs need special treatment

An ordinary differential equation (ODE) only needs the current state `h(u)` to advance the solution. A DDE additionally needs the *past* state `h(u − 1)`, which is not yet part of the solution when we start integrating. The standard fix is to supply a **history function** `phi` that defines `h` on the interval `[−1, 0]` before integration begins.

### Method of Steps

The idea is to convert the DDE into a sequence of ODEs, one per unit interval:

- On `[0, 1]`: the delayed term `h(u−1)` falls entirely inside `[−1, 0]`, so it is just `phi(u−1)` — a *known* function. The DDE becomes a plain ODE.
- On `[1, 2]`: the delayed term falls inside `[0, 1]`, which was just solved. Again a known function.
- Repeat for `[2, 3]`, `[3, 4]`, …

In practice we do **not** implement these intervals separately. Instead we maintain a single array of all computed values and look up `h(u−1)` by **linear interpolation** at each step.

### Forward Euler discretisation

With step size `dt`, the update rule at grid point `u_k` is:

```
h(u_{k+1}) = h(u_k) + dt * f(u_k)
```

where the right-hand side is:

```
f(u_k) = −ln(2) · (h(u_k) + 1) · h(u_k − 1)
```

The delayed value `h(u_k − 1)` is obtained by linearly interpolating the already-computed portion of the solution array.

### Grid alignment

To avoid floating-point drift when computing `u_k − 1`, the delay is snapped to the nearest integer multiple of `dt` at the start. This ensures the delayed index always lands exactly on a grid point or between two known grid points.

## Code Structure Overview

The notebook contains three functions and a main experiment block.

| Component | Role |
|---|---|
| `solve_dde_forward_euler(phi, U, dt, delay)` | Core solver — builds the grid, runs the Forward Euler loop, returns `(u_all, h)` |
| `phi_const(c)` | Helper — produces a constant history function `phi(u) = c` |
| `make_plot(u, h, title, filename, xlim)` | Utility — plots a solution curve and saves it as a PNG |

**Four experiments are run**, varying the initial history and the step size:

| Figure | History `phi(u)` | Step size `dt` | Purpose |
|---|---|---|---|
| 4.1 | 1.0 | 0.01 | Baseline: large positive initial value |
| 4.2 | −0.5 | 0.01 | Approach from the negative side |
| 4.3 | 1.0 | 0.02 | Coarser step — observe accuracy loss |
| 4.4 | 1.0 | 0.01 | Fine step — reference solution for comparison |

Figures 4.3 and 4.4 together illustrate how step-size reduction improves numerical accuracy.

---
## 1. Imports and Constants

We rely on three standard libraries:

- **NumPy** — array operations and `linspace` / `concatenate` for grid construction
- **math** — scalar `log` for the exact value of ln(2)
- **Matplotlib** — plotting and saving figures

`LN2` is computed once here and reused throughout, rather than calling `math.log(2)` inside the tight integration loop.

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

# Precompute ln(2) once to avoid repeated calls inside the loop
LN2 = math.log(2)

---
## 2. Core Solver: `solve_dde_forward_euler`

This function contains the entire numerical procedure in five steps:

1. **Align the delay** to the nearest integer multiple of `dt`, preventing floating-point drift when looking up past values.
2. **Build the history grid** `[−delay, 0]` and sample the history function `phi` on it.
3. **Build the forward grid** `(0, U]` and allocate the solution array (initialised to zero, filled in step 5).
4. **Concatenate** history and forward arrays into a single pair `(u_all, h)` so that all past values are always at hand.
5. **March forward** with Forward Euler: at each step retrieve `h(u_k − 1)` via linear interpolation of already-computed values, then apply the update rule.

The inner helper `h_at(tau, idx_current)` performs the linear interpolation and deliberately restricts itself to the computed prefix of the solution array, enforcing causality.

In [ ]:
def solve_dde_forward_euler(phi, U=20.0, dt=0.01, delay=1.0):
    """
    Solve the DDE:
        h'(u) = -(ln2) * (h(u) + 1) * h(u - 1),  u in [0, U]
    with history (initial condition):
        h(u) = phi(u),  u in [-delay, 0]

    Args:
        phi   : callable defining the history function on [-delay, 0]
        U     : right endpoint of the solution interval
        dt    : time step size
        delay : delay value (default 1.0)

    Returns:
        u_all : full time grid (history segment + forward segment)
        h     : array of solution values at each grid point
    """

    # ----------------------------------------------------------
    # Step 1: Align the delay to the nearest integer multiple of dt.
    # This prevents floating-point drift when looking up delayed values.
    # ----------------------------------------------------------
    m = int(round(delay / dt))   # number of grid points in the history segment
    delay = m * dt               # recompute exact delay after alignment

    # ----------------------------------------------------------
    # Step 2: Build the history grid [-delay, 0] with m+1 points,
    # and evaluate the history function phi at each node.
    # ----------------------------------------------------------
    u_hist = np.linspace(-delay, 0.0, m + 1)               # time nodes for history
    h_hist = np.array([phi(u) for u in u_hist], dtype=float)  # h values from phi

    # ----------------------------------------------------------
    # Step 3: Build the forward grid (0, U] with n_steps points.
    # Starts at dt (not 0, since 0 is already the last history node).
    # ----------------------------------------------------------
    n_steps = int(round(U / dt))          # total number of forward steps
    u_fwd = np.linspace(dt, U, n_steps)   # time nodes for the forward segment

    # ----------------------------------------------------------
    # Step 4: Concatenate history and forward arrays into one pair.
    # u_all: full time axis (m+1+n_steps points total)
    # h    : solution array; forward portion initialised to 0, filled in the loop
    # ----------------------------------------------------------
    u_all = np.concatenate([u_hist, u_fwd])
    h = np.concatenate([h_hist, np.zeros(n_steps)])

    # ----------------------------------------------------------
    # Helper: linearly interpolate h at time tau,
    # using only values already computed up to index idx_current.
    # This enforces causality — no future values are used.
    # ----------------------------------------------------------
    def h_at(tau, idx_current):
        """
        Return h(tau) by linear interpolation over the computed prefix
        u_all[0 : idx_current+1], h[0 : idx_current+1].
        """
        prefix_u = u_all[:idx_current + 1]   # known time nodes
        prefix_h = h[:idx_current + 1]        # corresponding known solution values

        # Find insertion index j such that prefix_u[j-1] <= tau < prefix_u[j]
        j = np.searchsorted(prefix_u, tau)

        # Clamp to left boundary
        if j <= 0:
            return prefix_h[0]
        # Clamp to right boundary
        if j >= len(prefix_u):
            return prefix_h[-1]

        # Linear interpolation between the two bracketing nodes
        u0, u1 = prefix_u[j - 1], prefix_u[j]   # left and right time nodes
        h0, h1 = prefix_h[j - 1], prefix_h[j]   # left and right h values
        # h(tau) ≈ h0 + (h1 - h0) * (tau - u0) / (u1 - u0)
        return h0 + (h1 - h0) * (tau - u0) / (u1 - u0)

    # ----------------------------------------------------------
    # Step 5: Forward Euler main loop.
    # k ranges from m (first forward index) to m+n_steps-1 (last).
    # At each step: h[k+1] = h[k] + dt * f(h[k], h[k - delay])
    # ----------------------------------------------------------
    for k in range(m, m + n_steps):
        u_k = u_all[k]          # current time u_k
        h_k = h[k]              # current solution value h(u_k)

        # Retrieve the delayed solution value h(u_k - delay) via interpolation
        h_delay = h_at(u_k - delay, k)

        # Evaluate the right-hand side: f = -(ln2) * (h(u) + 1) * h(u - delay)
        rhs = -LN2 * (h_k + 1.0) * h_delay

        # Forward Euler update: h(u_{k+1}) = h(u_k) + dt * f
        h[k + 1] = h_k + dt * rhs

    # Return the complete time grid and the solution array
    return u_all, h

---
## 3. Helper Functions

Two small utilities support the solver:

**`phi_const(c)`** is a *factory function*: it takes a constant `c` and returns a function `phi(u) = c`. This is a convenient way to express "the solution was identically equal to `c` for all past time" without writing a new function by hand for each experiment.

**`make_plot(u, h, ...)`** wraps the repetitive Matplotlib boilerplate — creating a figure, plotting the curve, adding axis labels, saving, and closing — into a single call. The `y = 0` reference line is included deliberately so that zero-crossings and convergence to the equilibrium `h* = 0` are immediately visible.

In [ ]:
def phi_const(c):
    """
    Factory function that returns a constant history function phi(u) = c.
    Used to set up constant initial conditions (e.g. phi = 1 or phi = -0.5).
    """
    return lambda u: c   # ignore u, always return the constant c


def make_plot(u, h, title, filename, xlim=(-1, 20)):
    """
    Plot the numerical solution and save it as a PNG file.

    Args:
        u        : time node array
        h        : solution value array
        title    : figure title string
        filename : output file path
        xlim     : x-axis display range, default (-1, 20)
    """
    plt.figure()                   # open a new figure window
    plt.plot(u, h)                 # plot the solution curve h(u)
    plt.axhline(0, linewidth=1)    # draw a y=0 reference line to highlight zero crossings
    plt.xlim(*xlim)                # set x-axis limits (unpack the tuple)
    plt.xlabel("u")                # label the horizontal axis
    plt.ylabel("h(u)")             # label the vertical axis
    plt.title(title)               # set the figure title
    plt.tight_layout()             # auto-adjust margins so labels are not clipped
    plt.savefig(filename, dpi=200) # save at 200 DPI for high-resolution output
    plt.close()                    # close the figure to free memory

---
## 4. Numerical Experiments

We run four experiments and save the results as figures.

**Varying the initial history** (Figures 4.1 and 4.2, both with `dt = 0.01`):

- `phi = 1` starts the trajectory well above the equilibrium `h* = 0` and shows oscillatory decay back toward zero.
- `phi = -0.5` starts between the two equilibria `h* = -1` and `h* = 0`, revealing how the trajectory approaches zero from the negative side.

**Varying the step size** (Figures 4.3 and 4.4, both with `phi = 1`):

- `dt = 0.02` (Figure 4.3) uses a coarser grid — fewer points per unit interval, larger truncation error.
- `dt = 0.01` (Figure 4.4) halves the step size, reducing the per-step error by a factor of roughly 2 (first-order method) and producing a visibly smoother, more accurate solution.

Comparing 4.3 and 4.4 side by side is a standard **step-size convergence check**: if the solutions look similar, both step sizes are in the convergent regime; if they differ noticeably, the coarser grid is under-resolved.

In [ ]:
U = 20.0   # solve on [0, 20]

# ---- Figure 4.1: constant history h(u)=1, step size dt=0.01 ----
# Large positive initial value; observe how the solution decays toward equilibrium
u, h = solve_dde_forward_euler(phi_const(1.0), U=U, dt=0.01)
make_plot(u, h,
          "Numerical solution, history h(u)=1 on [-1,0], dt=0.01",
          "fig_4_1.png", xlim=(-1, U))

# ---- Figure 4.2: constant history h(u)=-0.5, step size dt=0.01 ----
# Initial value in (-1, 0); observe convergence to equilibrium from the negative side
u, h = solve_dde_forward_euler(phi_const(-0.5), U=U, dt=0.01)
make_plot(u, h,
          "Numerical solution, history h(u)=-0.5 on [-1,0], dt=0.01",
          "fig_4_2.png", xlim=(-1, U))

# ---- Figure 4.3: constant history h(u)=1, coarser step size dt=0.02 ----
# Compare with Figure 4.4 to show the effect of doubling the step size
u, h = solve_dde_forward_euler(phi_const(1.0), U=U, dt=0.02)
make_plot(u, h,
          "Step size dt=0.02 (history h(u)=1 on [-1,0])",
          "fig_4_3.png", xlim=(-1, U))

# ---- Figure 4.4: constant history h(u)=1, finer step size dt=0.01 ----
# Pair with Figure 4.3 to illustrate improved accuracy with halved step size
u, h = solve_dde_forward_euler(phi_const(1.0), U=U, dt=0.01)
make_plot(u, h,
          "Step size dt=0.01 (history h(u)=1 on [-1,0])",
          "fig_4_4.png", xlim=(-1, U))

print("Saved figures: fig_4_1.png, fig_4_2.png, fig_4_3.png, fig_4_4.png")

---
## 5. 系统性初始值实验——旁证（Circumstantial Evidence）

### 为什么图 4.4 的历史段是常数 1？为什么不是 0.8？不是 -0.2？

这是**初始条件的选择**，而非方程本身决定的。DDE 的历史函数 φ(u) 扮演 ODE 初始值的角色——我们选了 φ = 1，也可以选任何其他值。

老师提出的核心问题是：**无论从哪个初始值出发，解 h(u) 是否都趋向 0？**

如果对多个不同的初始值都观察到 h(u) → 0，这就是"circumstantial evidence"（旁证），说明 h*(u) = 0 是一个**吸引平衡点**，真实解趋向 1/ln(x)。

### 测试的初始值

我们系统地测试 11 个初始值，覆盖多个不同区间：

| φ | 说明 |
|---|------|
| −0.99 | 紧靠非平凡平衡点 h* = −1 |
| −0.8 | 深负值区间 |
| −0.5 | 已有（图 4.2） |
| −0.2 | 老师特别提问的值 |
| 0.0 | 老师要求；**平凡平衡点**（h' = 0，解恒为 0） |
| 0.3 | 正值区间补充 |
| 0.7 | 老师要求 |
| 0.8 | 老师特别提问的值 |
| 1.0 | 已有（图 4.1） |
| 2.0 | 超出 [0,1] 的大正值 |
| 5.0 | 极大正值，测试稳健性 |

所有实验均使用 Forward Euler，dt = 0.01，U = 20。

In [ ]:
def make_overlay_plot(runs, title, filename, xlim=(-1, 20)):
    """
    Plot multiple solution curves on the same axes and save as PNG.

    Args:
        runs     : list of (u, h, label) tuples
        title    : figure title string
        filename : output file path
        xlim     : x-axis display range
    """
    plt.figure(figsize=(10, 5))
    for u, h, label in runs:
        plt.plot(u, h, label=label)
    plt.axhline(0, color='k', linewidth=1, linestyle='--')   # equilibrium h*=0
    plt.axhline(-1, color='gray', linewidth=0.8, linestyle=':')  # equilibrium h*=-1
    plt.xlim(*xlim)
    plt.xlabel("u")
    plt.ylabel("h(u)")
    plt.title(title)
    plt.legend(loc='upper right', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()


# --- Individual plots for teacher-specified values ---

# Fig 5.1: phi = 0 (trivial equilibrium — solution stays at 0 exactly)
u, h = solve_dde_forward_euler(phi_const(0.0), U=20.0, dt=0.01)
make_plot(u, h,
          r"$\phi=0.0$: trivial equilibrium — $h(u)\equiv 0$ for all $u\geq 0$",
          "figs/fig_5_1.png", xlim=(-1, 20))

# Fig 5.2: phi = 0.7 (teacher-specified)
u, h = solve_dde_forward_euler(phi_const(0.7), U=20.0, dt=0.01)
make_plot(u, h,
          r"$\phi=0.7$: solution decays from 0.7 toward $h^*=0$",
          "figs/fig_5_2.png", xlim=(-1, 20))

# Fig 5.3: phi = -0.2 and phi = 0.8 (teacher-questioned values)
u1, h1 = solve_dde_forward_euler(phi_const(-0.2), U=20.0, dt=0.01)
u2, h2 = solve_dde_forward_euler(phi_const(0.8),  U=20.0, dt=0.01)
make_overlay_plot(
    [(u1, h1, r"$\phi=-0.2$"), (u2, h2, r"$\phi=0.8$")],
    r"Why not $\phi=-0.2$ or $\phi=0.8$? Both converge to $h^*=0$",
    "figs/fig_5_3.png"
)

print("Saved fig_5_1.png, fig_5_2.png, fig_5_3.png")

In [ ]:
# --- Fig 5.4: Overlay of all 11 initial values — the strongest visual evidence ---
phi_values = [-0.99, -0.8, -0.5, -0.2, 0.0, 0.3, 0.7, 0.8, 1.0, 2.0, 5.0]

runs_all = []
for phi_c in phi_values:
    u_i, h_i = solve_dde_forward_euler(phi_const(phi_c), U=20.0, dt=0.01)
    runs_all.append((u_i, h_i, rf"$\phi={phi_c}$"))

make_overlay_plot(
    runs_all,
    r"Circumstantial evidence: all 11 initial values converge to $h^*(u)=0$",
    "figs/fig_5_4.png"
)

print("Saved fig_5_4.png  (overlay of 11 initial conditions)")

---
## 6. 更高精度方法：RK4 + Hermite 插值

### 老师的问题

> "Forward Euler may accumulate errors quickly as u gets bigger. Is there a method more precise than Euler? Runge-Kutta?"

老师提示的是 Runge-Kutta（RK4）方法。**但对 DDE 来说，直接套用 RK4 并不简单。**

---

### 为什么 "换成 RK4" 不是一句话的事？

对 **ODE** `y' = f(t, y)`，RK4 四级公式的误差精心配置，互相消去到 5 阶，全局精度为 **4 阶**（误差 ∝ dt⁴）。

对 **DDE** `h'(u) = f(h(u), h(u−1))`，RK4 的中间阶段需要延迟项：

| 阶段 | 当前时刻 | 需要的延迟值 |
|------|---------|------------|
| k₁ | u_k | h(u_k − 1) ← 格点，已有 |
| k₂ | u_k + dt/2 | h(u_k + dt/2 − 1) ← **非格点** |
| k₃ | u_k + dt/2 | h(u_k + dt/2 − 1) ← **非格点** |
| k₄ | u_k + dt | h(u_k + dt − 1) ← 非格点 |

对于非格点时刻，必须用插值来估计 h 的值。**插值精度直接决定整体方法的阶数：**

$$\text{全局精度} = \min(\text{RK4 阶数},\; \text{插值阶数})$$

- **线性插值**：误差 ≈ h''(τ)·(dt)²/8，即 O(dt²)  → 整体只有 **2 阶**
- **Hermite 三次插值**：误差 ≈ O(dt⁴)，与 RK4 匹配 → 整体达到 **3 阶**

---

### Hermite 三次插值

在每个格点 u_k，同时存储 h(u_k) 和 **h'(u_k)**。h'(u_k) 从 DDE 右端项直接得到：

$$h'(u_k) = -\ln 2 \cdot (h(u_k) + 1) \cdot h(u_k - 1)$$

设 τ 落在区间 [u_j, u_{j+1}]，令 t = (τ − u_j)/(u_{j+1} − u_j) ∈ [0,1]，
Hermite 三次公式为：

$$h(\tau) \approx h_j(1+2t)(1-t)^2 + h_{j+1}t^2(3-2t) + h'_j\,\Delta u\,t(1-t)^2 + h'_{j+1}\,\Delta u\,t^2(t-1)$$

此公式精确通过两端点的函数值和导数值，插值误差为 O(dt⁴)。

---

### 精度总结

| 方法 | 延迟插值 | 全局精度阶 |
|------|---------|-----------|
| Forward Euler | 线性（O(dt²)）| **1 阶** |
| RK4 + 线性插值 | 线性（O(dt²)）| **2 阶** |
| **RK4 + Hermite** | Hermite（O(dt⁴)）| **3 阶** |

In [ ]:
def solve_dde_rk4_hermite(phi, phi_prime, U=20.0, dt=0.01, delay=1.0):
    """
    Solve the DDE h'(u) = -ln2*(h(u)+1)*h(u-1) using RK4 with Hermite cubic
    interpolation for delayed values.

    Hermite interpolation uses both h values and h' values (derivatives) at
    each grid point, achieving O(dt^4) interpolation accuracy and O(dt^3)
    global accuracy — significantly better than Forward Euler (O(dt) global).

    Args:
        phi        : callable, history function on [-delay, 0]
        phi_prime  : callable, derivative of phi (use lambda u: 0 for constant phi)
        U          : right endpoint of the solution interval
        dt         : time step size
        delay      : delay value (default 1.0)

    Returns:
        u_all : full time grid
        h     : solution values
        hp    : derivative h'(u) at each grid point
    """
    # --- Step 1: Align delay ---
    m = int(round(delay / dt))
    delay = m * dt

    # --- Step 2: Build history grid and sample phi, phi' ---
    u_hist  = np.linspace(-delay, 0.0, m + 1)
    h_hist  = np.array([phi(u)       for u in u_hist], dtype=float)
    hp_hist = np.array([phi_prime(u) for u in u_hist], dtype=float)

    # --- Step 3: Build forward grid ---
    n_steps = int(round(U / dt))
    u_fwd   = np.linspace(dt, U, n_steps)

    # --- Step 4: Concatenate into single arrays ---
    u_all = np.concatenate([u_hist, u_fwd])
    h     = np.concatenate([h_hist,  np.zeros(n_steps)])
    hp    = np.concatenate([hp_hist, np.zeros(n_steps)])  # derivative storage

    # ------------------------------------------------------------------
    # Hermite cubic interpolation helper.
    # Given tau and the already-computed prefix up to idx_current,
    # returns h(tau) using cubic Hermite interpolation over the bracket.
    # Interpolation error is O(dt^4), matching RK4's local truncation error.
    # ------------------------------------------------------------------
    def h_at(tau, idx_current):
        prefix_u  = u_all[:idx_current + 1]
        prefix_h  = h[:idx_current + 1]
        prefix_hp = hp[:idx_current + 1]

        j = np.searchsorted(prefix_u, tau)

        if j <= 0:
            return prefix_h[0]
        if j >= len(prefix_u):
            return prefix_h[-1]

        u0, u1 = prefix_u[j - 1], prefix_u[j]
        h0, h1 = prefix_h[j - 1], prefix_h[j]
        d0, d1 = prefix_hp[j - 1], prefix_hp[j]

        dt_loc = u1 - u0            # local interval width
        t = (tau - u0) / dt_loc    # normalised coordinate in [0, 1]

        # Hermite cubic basis functions
        p0 = (1.0 + 2.0*t) * (1.0 - t)**2   # h0 weight
        p1 = t**2 * (3.0 - 2.0*t)            # h1 weight
        q0 = t * (1.0 - t)**2                # d0 weight (scaled by dt_loc)
        q1 = t**2 * (t - 1.0)               # d1 weight (scaled by dt_loc)

        return h0*p0 + h1*p1 + d0*dt_loc*q0 + d1*dt_loc*q1

    # ------------------------------------------------------------------
    # DDE right-hand side: f(h_val, h_del) = -ln2*(h_val+1)*h_del
    # This is also h'(u) at the current time, used to populate hp[].
    # ------------------------------------------------------------------
    def rhs(h_val, h_del):
        return -LN2 * (h_val + 1.0) * h_del

    # --- Step 5: RK4 main loop ---
    for k in range(m, m + n_steps):
        u_k = u_all[k]
        h_k = h[k]

        # Store h'(u_k) — needed by Hermite interpolation in future steps
        hp[k] = rhs(h_k, h_at(u_k - delay, k))

        # RK4 four stages; delayed values obtained via Hermite interpolation
        k1 = hp[k]   # == rhs(h_k, h_at(u_k - delay, k)), already computed
        k2 = rhs(h_k + 0.5*dt*k1, h_at(u_k + 0.5*dt - delay, k))
        k3 = rhs(h_k + 0.5*dt*k2, h_at(u_k + 0.5*dt - delay, k))
        k4 = rhs(h_k +     dt*k3, h_at(u_k +     dt - delay, k))

        h[k + 1] = h_k + (dt / 6.0) * (k1 + 2.0*k2 + 2.0*k3 + k4)

    # Store derivative at the final point
    hp[m + n_steps] = rhs(h[m + n_steps],
                           h_at(u_all[m + n_steps] - delay, m + n_steps))

    return u_all, h, hp

### 6.1 收敛率研究（Convergence Rate Study）

用极细步长（dt = 0.001）的 RK4+Hermite 解作为**参考解**，
衡量不同 dt 下 Forward Euler 和 RK4+Hermite 的 RMS 误差。

在 log-log 图上，误差 ∝ dt^p 对应斜率 p 的直线：
- Forward Euler 应显示斜率 ≈ 1
- RK4 + Hermite 应显示斜率 ≈ 3

In [ ]:
from scipy.interpolate import interp1d

# ---------------------------------------------------------
# Reference solution: RK4+Hermite with very fine dt=0.001
# ---------------------------------------------------------
phi_zero_deriv = lambda u: 0.0   # derivative of any constant history

print("Computing reference solution (dt=0.001) ...")
u_ref, h_ref, _ = solve_dde_rk4_hermite(phi_const(1.0), phi_zero_deriv,
                                          U=20.0, dt=0.001)
# Only compare on u >= 0 (the forward segment)
mask_ref = u_ref >= 0.0
u_ref_fwd = u_ref[mask_ref]
h_ref_fwd = h_ref[mask_ref]
ref_interp = interp1d(u_ref_fwd, h_ref_fwd, kind='cubic', bounds_error=False,
                       fill_value=(h_ref_fwd[0], h_ref_fwd[-1]))

# ---------------------------------------------------------
# Convergence study: test Euler and RK4+Hermite at several dt
# ---------------------------------------------------------
dt_values = [0.16, 0.08, 0.04, 0.02, 0.01, 0.005]
errs_euler = []
errs_rk4   = []

for dt_test in dt_values:
    # Forward Euler
    u_e, h_e = solve_dde_forward_euler(phi_const(1.0), U=20.0, dt=dt_test)
    mask_e = u_e >= 0.0
    h_e_ref = ref_interp(u_e[mask_e])
    errs_euler.append(np.sqrt(np.mean((h_e[mask_e] - h_e_ref)**2)))

    # RK4 + Hermite
    u_r, h_r, _ = solve_dde_rk4_hermite(phi_const(1.0), phi_zero_deriv,
                                          U=20.0, dt=dt_test)
    mask_r = u_r >= 0.0
    h_r_ref = ref_interp(u_r[mask_r])
    errs_rk4.append(np.sqrt(np.mean((h_r[mask_r] - h_r_ref)**2)))

    print(f"  dt={dt_test:.3f}  Euler RMS={errs_euler[-1]:.2e}  RK4+H RMS={errs_rk4[-1]:.2e}")

# ---------------------------------------------------------
# Log-log convergence plot with reference slope lines
# ---------------------------------------------------------
dt_arr   = np.array(dt_values)
err_e    = np.array(errs_euler)
err_r    = np.array(errs_rk4)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(dt_arr, err_e, 'o-', color='steelblue',  label='Forward Euler')
ax.loglog(dt_arr, err_r, 's-', color='darkorange', label='RK4 + Hermite')

# Reference slope lines anchored at the coarsest dt
c1 = err_e[0] / dt_arr[0]**1
c3 = err_r[0] / dt_arr[0]**3
ax.loglog(dt_arr, c1 * dt_arr**1, 'k--', alpha=0.4, label=r'$O(\Delta t^1)$  slope 1')
ax.loglog(dt_arr, c3 * dt_arr**3, 'r--', alpha=0.4, label=r'$O(\Delta t^3)$  slope 3')

ax.set_xlabel(r'Step size $\Delta t$', fontsize=12)
ax.set_ylabel('RMS error vs reference solution', fontsize=12)
ax.set_title('Convergence Study: Forward Euler vs RK4 + Hermite interpolation',
             fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('figs/fig_6_1.png', dpi=200)
plt.close()
print("\nSaved figs/fig_6_1.png  (convergence rate log-log plot)")

### 6.2 视觉对比：Euler vs RK4+Hermite 在较大步长下的精度差异

在 dt = 0.05（步长较大）时，两种方法的解同框展示。
Euler 会有明显漂移，而 RK4+Hermite 仍紧贴参考解。

In [ ]:
# Fig 6.2: Visual comparison — use dt=0.2 so Euler shows visible error accumulation.
# At this large step size, Euler's first-order truncation error is clearly visible,
# while RK4+Hermite (3rd order) still tracks the reference solution closely.
dt_compare = 0.2
u_e, h_e = solve_dde_forward_euler(phi_const(1.0), U=20.0, dt=dt_compare)
u_r, h_r, _ = solve_dde_rk4_hermite(phi_const(1.0), phi_zero_deriv,
                                      U=20.0, dt=dt_compare)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [2, 1]})

# Top panel: solution curves
ax = axes[0]
ax.plot(u_ref[u_ref >= -1], h_ref[u_ref >= -1], 'k-', linewidth=2,
        label='Reference (RK4+Hermite, dt=0.001)', alpha=0.5, zorder=1)
ax.plot(u_e, h_e, 'b--', linewidth=1.8, label=f'Forward Euler (dt={dt_compare})', zorder=2)
ax.plot(u_r, h_r, 'r-',  linewidth=1.8, label=f'RK4+Hermite (dt={dt_compare})', zorder=3)
ax.axhline(0, color='gray', linewidth=0.6, linestyle=':')
ax.set_xlim(-1, 20)
ax.set_ylabel('h(u)')
ax.set_title(f'Euler vs RK4+Hermite at dt={dt_compare}: visual accuracy comparison')
ax.legend(fontsize=10)

# Bottom panel: pointwise error |method - reference|
ref_e = ref_interp(u_e[u_e >= 0])
ref_r = ref_interp(u_r[u_r >= 0])
err_e_pt = np.abs(h_e[u_e >= 0] - ref_e)
err_r_pt = np.abs(h_r[u_r >= 0] - ref_r)

ax2 = axes[1]
ax2.semilogy(u_e[u_e >= 0], err_e_pt + 1e-16, 'b--', linewidth=1.5,
             label=f'|Euler − ref|')
ax2.semilogy(u_r[u_r >= 0], err_r_pt + 1e-16, 'r-', linewidth=1.5,
             label=f'|RK4+Hermite − ref|')
ax2.set_xlim(-1, 20)
ax2.set_xlabel('u')
ax2.set_ylabel('|error|  (log scale)')
ax2.set_title('Pointwise error vs reference solution')
ax2.legend(fontsize=10)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('figs/fig_6_2.png', dpi=200)
plt.close()
print("Saved figs/fig_6_2.png  (solution + error panel)")

### 6.3 长期运行（U = 50）：RK4+Hermite 验证所有初始值收敛

Forward Euler 在大 u 时误差积累，难以可靠地展示长期行为。
RK4+Hermite 精度高，适合延伸到更大的 U = 50，进一步证明
**所有初始值的解都在 u ≈ 10–15 附近收敛到 0 并长期稳定**。

In [ ]:
# Fig 6.3: RK4+Hermite, U=50, overlay of all initial values
phi_values_long = [-0.99, -0.8, -0.5, -0.2, 0.0, 0.3, 0.7, 0.8, 1.0, 2.0, 5.0]
runs_long = []
for phi_c in phi_values_long:
    u_i, h_i, _ = solve_dde_rk4_hermite(phi_const(phi_c), phi_zero_deriv,
                                          U=50.0, dt=0.01)
    runs_long.append((u_i, h_i, rf"$\phi={phi_c}$"))

make_overlay_plot(
    runs_long,
    r"RK4+Hermite, $U=50$: long-run convergence for all initial values",
    "figs/fig_6_3.png",
    xlim=(-1, 50)
)
print("Saved figs/fig_6_3.png  (RK4+Hermite long-run overlay)")

---
## 7. 提升浮点精度（Increasing Length of Numerical Values）

老师提到两种提升数值精度的方法：

**方法一：减小步长 dt**（已在第 4 节和第 6.1 节中演示）

- dt 越小，每步误差越小，收敛率图清晰地量化了这种改进。

**方法二：使用更高精度的浮点数**

标准 Python/NumPy 浮点数 (`float64`) 使用 64 位，有效十进制精度约 **15~16 位**。
`numpy.longdouble` 在 x86 系统上使用 **80 位扩展精度**，有效十进制精度约 **18~19 位**。

这减少了每次算术运算的舍入误差，在超长时间积分（u ≫ 20）时可积累显著差异。

In [ ]:
# Compare float64 vs longdouble precision
print("=== Standard float64 ===")
fi64 = np.finfo(np.float64)
print(f"  Bits:                  {fi64.bits}")
print(f"  Significant digits:    ~{fi64.precision}")
print(f"  Machine epsilon:       {fi64.eps:.3e}")

print()
print("=== numpy.longdouble (80-bit extended on x86) ===")
fild = np.finfo(np.longdouble)
print(f"  Bits:                  {fild.bits}")
print(f"  Significant digits:    ~{fild.precision}")
print(f"  Machine epsilon:       {fild.eps:.3e}")

print()
print("To use longdouble in solve_dde_rk4_hermite, change array dtype:")
print("  h_hist  = np.array([phi(u) for u in u_hist], dtype=np.longdouble)")
print("  hp_hist = np.array([phi_prime(u) for u in u_hist], dtype=np.longdouble)")
print("  h  = np.concatenate([h_hist,  np.zeros(n_steps, dtype=np.longdouble)])")
print("  hp = np.concatenate([hp_hist, np.zeros(n_steps, dtype=np.longdouble)])")
print()
print("For U=20 with dt=0.01, float64 rounding is negligible (~1e-14 per step).")
print("For very long integrations (U >> 50), longdouble provides a meaningful safety margin.")